# Hardware Doctor Live Diagnostic

This notebook runs the new AWR2944 hardware doctor to verify your system setup and physical connections. It performs diagnostic checks without mutating any hardware state or executing captures.

In [ ]:
from awr2944_dca.lab import RadarProject
from pathlib import Path
import os

LIVE_PROJECT_ROOT = Path(r"C:\Users\khams008\Documents\awr2944-live-project")

if LIVE_PROJECT_ROOT.exists():
    project = RadarProject.open(LIVE_PROJECT_ROOT)
else:
    project = RadarProject.create_at(LIVE_PROJECT_ROOT, git_init=False)
    project.config.local.com_port = "COM8"
    project.config.local.aux_com_port = "COM7"
    project.config.local.host_ip = "192.168.33.30"
    project.config.portable.dca_ip = "192.168.33.180"
    project.config.portable.config_port = 4096
    project.config.portable.data_port = 4098
    project.config.local.dca_control_exe = r"C:\ti\mmwave_studio_03_01_04_04\mmWaveStudio\PostProc\DCA1000EVM_CLI_Control.exe"
    project.config.local.dca_record_exe = r"C:\ti\mmwave_studio_03_01_04_04\mmWaveStudio\PostProc\DCA1000EVM_CLI_Record.exe"
    project.config.local.cf_json_path = r"C:\ti\mmwave_studio_03_01_04_04\mmWaveStudio\PostProc\cf.json"
    project.config.save()

print(f"Selected project root: {project.root}")
print("Project Tree:")
for p in project.root.rglob('*'):
    print(f"  {p.relative_to(project.root)}")


## 1. Verify open_here() Behavior
Note that `open_here()` searches from the process CWD, not from the physical notebook-file location. This distinction matters in IDEs like VS Code and Jupyter, which may execute from a workspace root instead of the notebook's folder.

In [ ]:
original_cwd = os.getcwd()
try:
    os.chdir(LIVE_PROJECT_ROOT)
    project_here = RadarProject.open_here()
    assert project_here.root == LIVE_PROJECT_ROOT, "open_here() failed to return the same root"
    print("open_here() successfully returned the expected live project root based on CWD.")
finally:
    os.chdir(original_cwd)


## 2. Discover Hardware

First, we will scan the system to see what COM ports and network adapters are currently visible, regardless of configuration.

In [ ]:
discovery = project.hardware.discover()

print("=== Serial Ports (pyserial) ===")
for p in discovery.serial_ports:
    print(f"{p.port:8s} {p.name} (XDS110: {p.is_xds110})")

print("\n=== COM Port Roles (heuristics) ===")
for p in discovery.com_ports:
    print(f"{p.com:8s} -> {p.likely_role} (conf={p.confidence})")

print("\n=== Network Adapters ===")
for a in discovery.network_adapters:
    print(f"{a.get('InterfaceAlias', '')}: {a.get('IPAddress', '')}")

## 3. Run Hardware Doctor

Now we run the full diagnostic suite. This will check file structures, executables, network bindings, and attempt read-only probes of the UART and DCA APIs.

In [ ]:
report = project.doctor(include_hardware=True)
report.print()

## 4. Strict Verification

Uncomment the following line to ensure that an exception is raised if the system is not completely ready for capture.

In [ ]:
# report.raise_for_errors(strict=True)